<a href="https://colab.research.google.com/github/ThousandAI/PythonAI-010/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**專案：YouBike 借還車互動地圖**
================================
【怎麼寫】完成標「# TODO」的地方即可，其餘已寫好、直接執行。

功能：點地圖任一點 → 跳出最近 5 站 + 拉線 + 清單；
      可切換借車/還車。
資料來源：YouBike2.0 即時資訊（免金鑰）｜環境：Colab｜地圖：ipyleaflet


In [2]:
# ============================================================
# 【格 0】安裝套件（直接執行；裝完建議「執行階段→重新啟動並全部執行」）
# ============================================================
get_ipython().system('pip -q install ipyleaflet')

from google.colab import output
output.enable_custom_widget_manager()

# ============================================================
# 【格 1】抓 API + 資料清理（已幫你寫好，直接執行）
# ============================================================
import requests
import pandas as pd

URL = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"


def fetch_youbike():
    d = pd.DataFrame(requests.get(URL, timeout=10).json())

    rename = {}
    if "latitude" in d.columns:  rename["latitude"] = "lat"
    if "longitude" in d.columns: rename["longitude"] = "lng"
    for c in ["Quantity", "tot", "total", "totalstationcount"]:
        if c in d.columns:
            rename[c] = "capacity"
            break
    d = d.rename(columns=rename)

    # 數字欄位轉型（已幫你完成）
    for col in ["available_rent_bikes", "available_return_bikes", "capacity", "lat", "lng"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    d = d.dropna(subset=["lat", "lng"]).reset_index(drop=True)
    return d


df = fetch_youbike()
print("總站數：", len(df))

總站數： 1774


In [ ]:
# ============================================================
# 【格 2】距離公式 + 找最近站
# ============================================================
from math import radians, sin, cos, sqrt, atan2

WALK_KMH = 4.8


def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


def nearest(lat, lng, mode="借車", top=5):
    # TODO(1)：依 mode 選欄位（借車看 available_rent_bikes、還車看 available_return_bikes），
    #          只保留該欄位 > 0 的站，計算每站到 (lat,lng) 的距離與步行分鐘，
    #          由近到遠排序後回傳前 top 站，欄位需含 sna, sarea, avail, dist, walk, lat, lng。
    pass

In [ ]:
# ============================================================
# 【格 3】互動地圖
# ============================================================
from ipyleaflet import Map, Marker, CircleMarker, Polyline, AwesomeIcon
import ipywidgets as widgets
from IPython.display import display

m = Map(center=(25.0478, 121.5319), zoom=12, scroll_wheel_zoom=True)

# TODO(2)：把 df 裡「每個站」畫成地圖上的小圓點(CircleMarker)當底圖，
#          （迴圈跑 df.iterrows()，每站 m.add_layer(...)）


mode_toggle = widgets.ToggleButtons(options=["借車", "還車"], value="借車", description="模式:")
clear_btn = widgets.Button(description="清除", button_style="warning", icon="eraser")
info_box = widgets.HTML(value="<b>👉 滑鼠移到站點看車數；點地圖任一位置找最近 5 站</b>")

dynamic_layers = []

In [ ]:
def clear_dynamic(_=None):
    for layer in dynamic_layers:
        try:
            m.remove_layer(layer)
        except Exception:
            pass
    dynamic_layers.clear()
    info_box.value = "<b>已清除</b>"


def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lng = kwargs["coordinates"]
    clear_dynamic()

    mode = mode_toggle.value
    result = nearest(lat, lng, mode=mode, top=5)
    kind = "可借" if mode == "借車" else "可還"

    click_marker = Marker(location=(lat, lng),
                          icon=AwesomeIcon(name="user", marker_color="red"))
    m.add_layer(click_marker); dynamic_layers.append(click_marker)

    rows = []
    # TODO(3)：把 result 的每個站標成綠色 CircleMarker，從點擊處拉一條 Polyline 到該站，
    #          並把「站名、可借/可還幾台、距離、步行分鐘」加進 rows 清單。
    #          （記得每個新圖層都要 m.add_layer(...) 並 append 到 dynamic_layers）


    info_box.value = (f"<b>📍 最近、且{kind}的前 5 站（{mode}）</b><br>"
                      + "<br>".join(rows))


m.on_interaction(on_click)
clear_btn.on_click(clear_dynamic)


def on_mode_change(_):
    info_box.value = f"<b>已切到「{mode_toggle.value}」，請重新點一下地圖</b>"
mode_toggle.observe(on_mode_change, names="value")

display(widgets.HBox([mode_toggle, clear_btn]))
display(widgets.HBox([m, info_box]))